# Policy Gradient

**Reinforcement Learning Course** — Notebook 04

| Algorithm | Environment | Key idea |
|-----------|-------------|----------|
| REINFORCE | InvertedPendulum | Monte Carlo PG |
| PPO | Humanoid | Clipped surrogate · 2048 parallel envs |

In [1]:
!pip install -q brax flax optax plotly

  DEPRECATION: Building 'ml_collections' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'ml_collections'. Discussion can be found at https://github.com/pypa/pip/issues/6334
  error: subprocess-exited-with-error
  
  × Building wheel for mujoco (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [166 lines of output]
      running bdist_wheel
      running build
      running build_py
      creating build/lib.macosx-10.9-universal2-cpython-39/mujoco
      copying mujoco/viewer.py -> build/lib.macosx-10.9-universal2-cpython-39/mujoco
      copying mujoco/renderer.py -> build/lib.macosx-10.9-universal2-cpython-39/mujoco
      copying mujoco/renderer_test.py -> build/lib.macosx-10.9

In [2]:
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
import numpy as np
import plotly.graph_objects as go
from brax import envs
from brax.io import html
from IPython.display import HTML

print('devices:', jax.devices())

ModuleNotFoundError: No module named 'jax'

## Part 1 — REINFORCE

**Policy Gradient Theorem:**

$$\nabla_\theta J(\theta) = \underset{\substack{S_t \sim \rho^{\pi_\theta}_\gamma \\ A_t \sim \pi_\theta(\cdot \mid S_t)}}{\mathbb{E}}\!\left[\nabla_\theta \log \pi_\theta(A_t \mid S_t)\cdot G_t\right]$$

For continuous actions: $\pi_\theta(\cdot \mid s) = \mathcal{N}(\mu_\theta(s),\, e^{2\sigma})$, $\sigma$ a learned log-std.  
Rollout → compute $G_t$ → gradient step. One episode at a time.

In [ ]:
# ── Policy network (MLP → Gaussian mean) ──────────────────────────────────────
class PolicyNet(nn.Module):
    action_dim: int

    @nn.compact
    def __call__(self, s: jnp.ndarray) -> jnp.ndarray:
        x = nn.relu(nn.Dense(64)(s))
        x = nn.relu(nn.Dense(64)(x))
        return nn.Dense(self.action_dim)(x)


def gaussian_log_prob(
    mean: jnp.ndarray, log_std: jnp.ndarray, a: jnp.ndarray
) -> jnp.ndarray:
    var = jnp.exp(2.0 * log_std)
    return -0.5 * jnp.sum((a - mean) ** 2 / var + 2.0 * log_std + jnp.log(2.0 * jnp.pi))


# ── Rollout: collect (obs, actions, rewards) ───────────────────────────────────
def rollout(
    params: dict,
    log_std: jnp.ndarray,
    net: PolicyNet,
    env,
    key: jax.Array,
    max_steps: int = 1000,
    gamma: float = 0.99,
) -> tuple[jnp.ndarray, jnp.ndarray, jnp.ndarray, float]:
    key, k0 = jax.random.split(key)
    state = env.reset(k0)

    obs_buf, act_buf, rew_buf = [], [], []
    for _ in range(max_steps):
        key, ka = jax.random.split(key)
        mean = net.apply(params, state.obs)
        a = mean + jnp.exp(log_std) * jax.random.normal(ka, mean.shape)
        obs_buf.append(state.obs)
        act_buf.append(a)
        state = env.step(state, a)
        rew_buf.append(float(state.reward))
        if float(state.done) > 0.5:
            break

    # Discounted returns (normalised)
    G, returns = 0.0, []
    for r in reversed(rew_buf):
        G = r + gamma * G
        returns.insert(0, G)
    returns = jnp.array(returns)
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)

    return (
        jnp.stack(obs_buf),
        jnp.stack(act_buf),
        returns,
        sum(rew_buf),
    )


# ── Loss and gradient ──────────────────────────────────────────────────────────
@jax.jit
def reinforce_loss(
    params: dict,
    log_std: jnp.ndarray,
    net: PolicyNet,
    obs: jnp.ndarray,
    actions: jnp.ndarray,
    returns: jnp.ndarray,
) -> jnp.ndarray:
    means = jax.vmap(lambda s: net.apply(params, s))(obs)
    log_probs = jax.vmap(gaussian_log_prob)(means, jnp.broadcast_to(log_std, means.shape), actions)
    return -jnp.mean(log_probs * returns)

In [ ]:
np.random.seed(42)
key = jax.random.PRNGKey(42)

env_pend = envs.get_environment('inverted_pendulum')
obs_dim  = env_pend.observation_size
act_dim  = env_pend.action_size

net     = PolicyNet(action_dim=act_dim)
key, k0 = jax.random.split(key)
params  = net.init(k0, jnp.zeros(obs_dim))
log_std = jnp.full((act_dim,), -0.5)

optimizer = optax.adam(3e-4)
opt_state = optimizer.init((params, log_std))

ep_returns: list[float] = []

for ep in range(500):
    key, k_ep = jax.random.split(key)
    obs, actions, returns, total_r = rollout(params, log_std, net, env_pend, k_ep)
    ep_returns.append(total_r)

    grads = jax.grad(reinforce_loss, argnums=(0, 1))(
        params, log_std, net, obs, actions, returns
    )
    updates, opt_state = optimizer.update(grads, opt_state, (params, log_std))
    params, log_std = optax.apply_updates((params, log_std), updates)

    if ep % 50 == 0:
        print(f'ep {ep:4d}  return={total_r:.1f}')

print('Done.')

In [ ]:
def smooth(x: list[float], w: int = 20) -> np.ndarray:
    return np.convolve(x, np.ones(w) / w, mode='valid')

fig = go.Figure()
fig.add_trace(go.Scatter(
    y=ep_returns, mode='lines',
    line=dict(color='rgba(0,212,255,0.25)', width=1),
    name='raw',
))
fig.add_trace(go.Scatter(
    y=smooth(ep_returns), mode='lines',
    line=dict(color='rgba(0,212,255,1)', width=2),
    name='smoothed',
))
fig.update_layout(
    title='REINFORCE — InvertedPendulum',
    xaxis_title='episode', yaxis_title='return',
    template='plotly_dark', height=350,
)
fig.show()

In [ ]:
# Render one episode with the trained policy
key, k_r = jax.random.split(key)
state = env_pend.reset(k_r)
pipeline_states = [state.pipeline_state]

for _ in range(500):
    key, ka = jax.random.split(key)
    mean = net.apply(params, state.obs)
    a = mean  # deterministic at eval time
    state = env_pend.step(state, a)
    pipeline_states.append(state.pipeline_state)
    if float(state.done) > 0.5:
        break

HTML(html.render(env_pend.sys, pipeline_states))

## Part 2 — PPO on Humanoid

**Clipped surrogate** (sampled from old policy $\pi_{\theta_\text{old}}$):

$$\mathcal{L}(\theta) = \underset{\substack{S_t \sim d^{\pi_{\theta_\text{old}}} \\ A_t \sim \pi_{\theta_\text{old}}(\cdot \mid S_t)}}{\mathbb{E}}\!\left[\min\!\left(r_t(\theta)\,\hat{A}_t,\;\operatorname{clip}(r_t(\theta),\,1\!\pm\!\varepsilon)\,\hat{A}_t\right)\right]$$

where $r_t(\theta) = \dfrac{\pi_\theta(A_t \mid S_t)}{\pi_{\theta_\text{old}}(A_t \mid S_t)}$ and $\hat{A}_t$ is estimated via **GAE**.

Key: **2048 environments in parallel** via `jax.vmap` — Brax runs entirely on GPU.

In [ ]:
# ── Actor-Critic: shared trunk + two heads ─────────────────────────────────────
class ActorCritic(nn.Module):
    action_dim: int
    hidden: int = 256

    @nn.compact
    def __call__(self, s: jnp.ndarray) -> tuple[jnp.ndarray, jnp.ndarray]:
        x = nn.tanh(nn.Dense(self.hidden)(s))
        x = nn.tanh(nn.Dense(self.hidden)(x))
        mean  = nn.Dense(self.action_dim)(x)       # policy head
        value = nn.Dense(1)(x).squeeze(-1)         # value head
        return mean, value


def ac_log_prob(
    mean: jnp.ndarray, log_std: jnp.ndarray, a: jnp.ndarray
) -> jnp.ndarray:
    var = jnp.exp(2.0 * log_std)
    return -0.5 * jnp.sum((a - mean) ** 2 / var + 2.0 * log_std + jnp.log(2.0 * jnp.pi))

In [ ]:
def compute_gae(
    rewards: jnp.ndarray,
    values: jnp.ndarray,
    dones: jnp.ndarray,
    gamma: float = 0.99,
    lam: float = 0.95,
) -> tuple[jnp.ndarray, jnp.ndarray]:
    """Generalised Advantage Estimation — vectorised over T steps."""
    T = len(rewards)
    advantages = jnp.zeros(T)
    gae = 0.0
    for t in reversed(range(T)):
        next_val = values[t + 1] if t < T - 1 else 0.0
        mask = 1.0 - dones[t]
        delta = rewards[t] + gamma * next_val * mask - values[t]
        gae = delta + gamma * lam * mask * gae
        advantages = advantages.at[t].set(gae)
    returns = advantages + values
    return advantages, returns

In [ ]:
def collect_rollouts(
    params: dict,
    log_std: jnp.ndarray,
    ac: ActorCritic,
    states,
    env_v,
    key: jax.Array,
    n_steps: int = 64,
) -> dict:
    """Collect n_steps from N parallel envs."""
    obs_buf, act_buf, rew_buf, done_buf, val_buf, logp_buf = [], [], [], [], [], []

    for _ in range(n_steps):
        key, ka = jax.random.split(key)
        obs = states.obs  # (N, obs_dim)

        def step_env(o, k):
            mean, val = ac.apply(params, o)
            a = mean + jnp.exp(log_std) * jax.random.normal(k, mean.shape)
            lp = ac_log_prob(mean, log_std, a)
            return a, val, lp

        keys_n = jax.random.split(ka, obs.shape[0])
        actions, values, log_probs = jax.vmap(step_env)(obs, keys_n)

        states = env_v.step(states, actions)
        obs_buf.append(obs)
        act_buf.append(actions)
        rew_buf.append(states.reward)
        done_buf.append(states.done)
        val_buf.append(values)
        logp_buf.append(log_probs)

    return dict(
        obs    = jnp.stack(obs_buf),     # (T, N, obs_dim)
        actions= jnp.stack(act_buf),     # (T, N, act_dim)
        rewards= jnp.stack(rew_buf),     # (T, N)
        dones  = jnp.stack(done_buf),    # (T, N)
        values = jnp.stack(val_buf),     # (T, N)
        log_probs_old = jnp.stack(logp_buf),  # (T, N)
        states = states,
    )

In [ ]:
@partial(jax.jit, static_argnums=(2,))
def ppo_loss(
    params: dict,
    log_std: jnp.ndarray,
    ac: ActorCritic,
    obs: jnp.ndarray,
    actions: jnp.ndarray,
    advantages: jnp.ndarray,
    returns: jnp.ndarray,
    log_probs_old: jnp.ndarray,
    clip_eps: float = 0.2,
    c1: float = 0.5,
    c2: float = 0.01,
) -> jnp.ndarray:
    # Flatten (T*N, ...)
    obs_f  = obs.reshape(-1, obs.shape[-1])
    act_f  = actions.reshape(-1, actions.shape[-1])
    adv_f  = advantages.reshape(-1)
    ret_f  = returns.reshape(-1)
    old_lp = log_probs_old.reshape(-1)

    adv_f = (adv_f - adv_f.mean()) / (adv_f.std() + 1e-8)

    means, values = jax.vmap(lambda o: ac.apply(params, o))(obs_f)
    new_lp = jax.vmap(ac_log_prob)(
        means, jnp.broadcast_to(log_std, means.shape), act_f
    )

    ratio = jnp.exp(new_lp - old_lp)
    l_clip = jnp.mean(jnp.minimum(
        ratio * adv_f,
        jnp.clip(ratio, 1.0 - clip_eps, 1.0 + clip_eps) * adv_f,
    ))
    l_value   = jnp.mean((values - ret_f) ** 2)
    l_entropy = jnp.mean(0.5 * jnp.sum(2.0 * log_std + jnp.log(2.0 * jnp.pi * jnp.e), axis=-1))

    return -(l_clip - c1 * l_value + c2 * l_entropy)

In [ ]:
N_ENVS   = 2048
N_STEPS  = 64
N_EPOCHS = 4
N_ITERS  = 200

key = jax.random.PRNGKey(0)

env_h    = envs.get_environment('humanoid')
obs_dim  = env_h.observation_size
act_dim  = env_h.action_size

# Vectorised env (N_ENVS parallel)
env_v    = envs.wrappers.training.VmapWrapper(env_h)

ac       = ActorCritic(action_dim=act_dim)
key, k0  = jax.random.split(key)
params   = ac.init(k0, jnp.zeros(obs_dim))
log_std  = jnp.full((act_dim,), -0.5)

optimizer = optax.chain(
    optax.clip_by_global_norm(0.5),
    optax.adam(3e-4),
)
opt_state = optimizer.init((params, log_std))

# Reset all envs
key, k_reset = jax.random.split(key)
keys_reset   = jax.random.split(k_reset, N_ENVS)
states       = jax.vmap(env_h.reset)(keys_reset)

iter_returns: list[float] = []

for it in range(N_ITERS):
    key, k_roll = jax.random.split(key)
    batch = collect_rollouts(params, log_std, ac, states, env_v, k_roll, N_STEPS)
    states = batch.pop('states')

    # GAE per env
    advantages = jnp.zeros_like(batch['rewards'])
    returns    = jnp.zeros_like(batch['rewards'])
    for n in range(N_ENVS):
        adv_n, ret_n = compute_gae(
            batch['rewards'][:, n],
            batch['values'][:, n],
            batch['dones'][:, n],
        )
        advantages = advantages.at[:, n].set(adv_n)
        returns    = returns.at[:, n].set(ret_n)

    # PPO epochs
    for _ in range(N_EPOCHS):
        grads = jax.grad(ppo_loss, argnums=(0, 1))(
            params, log_std, ac,
            batch['obs'], batch['actions'],
            advantages, returns,
            batch['log_probs_old'],
        )
        updates, opt_state = optimizer.update(grads, opt_state, (params, log_std))
        params, log_std = optax.apply_updates((params, log_std), updates)

    mean_return = float(batch['rewards'].sum(0).mean())
    iter_returns.append(mean_return)
    if it % 20 == 0:
        print(f'iter {it:4d}  mean_return={mean_return:.1f}')

print('Done.')

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(
    y=iter_returns, mode='lines',
    line=dict(color='rgba(34,197,94,0.3)', width=1),
    name='raw',
))
fig.add_trace(go.Scatter(
    y=smooth(iter_returns, w=10), mode='lines',
    line=dict(color='rgba(34,197,94,1)', width=2),
    name='smoothed',
))
fig.update_layout(
    title='PPO — Humanoid (2048 parallel envs)',
    xaxis_title='iteration', yaxis_title='mean return per env',
    template='plotly_dark', height=350,
)
fig.show()

In [ ]:
# Render one episode of the trained Humanoid
key, k_r = jax.random.split(key)
state_h = env_h.reset(k_r)
pipeline_states = [state_h.pipeline_state]

for _ in range(500):
    mean_a, _ = ac.apply(params, state_h.obs)
    state_h = env_h.step(state_h, mean_a)  # deterministic
    pipeline_states.append(state_h.pipeline_state)
    if float(state_h.done) > 0.5:
        break

HTML(html.render(env_h.sys, pipeline_states))